In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
paper_STEP2_U25_k_summary_build.py
----------------------------------
Consolidate and publish paper-ready outputs (CSV + PDF) for the fixed "≤25 clusters" selection (STEP2_U25).

[Inputs] (auto-detected "latest" files):
  ROOT = /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25
  ├─ preprocessed_features_OH_U25/
  │    └─ *_cluster_k_summary_*.csv    ← use the NEWEST one
  └─ preprocessed_features_FP_U25/
       └─ *_cluster_k_summary_*.csv    ← use the NEWEST one

  Expected columns in each k-summary CSV:  mode, index, k

[Outputs] (written directly under ROOT):
  ├─ U25_k_summary_all_long_YYYYMMDD.csv      … long table: [dataset, mode, index, k]
  ├─ U25_k_summary_all_wide_YYYYMMDD.csv      … wide table: rows=index, cols=OH_top3,OH_cumeig,FP_top3,FP_cumeig
  ├─ Fig_U25_k_heatmap_YYYYMMDD.pdf           … heatmap: y=index, x=mode, fill=k (facets: dataset)
  └─ Fig_U25_k_bar_YYYYMMDD.pdf               … bar chart: k by index (facets: dataset × mode)

[Console]
  - Prints the wide table preview.

[Figure/Text policy]
  - All labels/titles in figures are **English only**.

[Assumptions]
  - STEP2 (≤25 clusters) has already been executed and generated per-dataset k-summary CSVs.
  - If multiple *_cluster_k_summary_*.csv exist, the most recently modified is used.

Diagram of folders (data flow)
ROOT (STEP2_U25)
├── preprocessed_features_OH_U25/        ─┐
│   └── *_cluster_k_summary_*.csv         ├─► (read latest) ┐
└── preprocessed_features_FP_U25/        ─┘                  ├─► merge → CSVs + PDFs under ROOT → console preview
    └── *_cluster_k_summary_*.csv        ─┘                 ┘

----------------------------------------------------------------
【日本語要約】
固定「25クラスター上限（STEP2_U25）」の決定結果（OH/FPの *_cluster_k_summary_*.csv）から、
論文用の CSV（long / wide）と PDF 図（ヒートマップ／棒グラフ）を ROOT 直下へ出力します。
図表の文字は英語のみです。複数CSVがある場合は更新日時が最新のものを自動選択します。
"""

from __future__ import annotations
import sys
from pathlib import Path
import datetime as dt

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# ---------- Paths ----------
BASE_DIR = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No")
ROOT_OUT = BASE_DIR / "20251026_under_25clusters" / "STEP2_U25"
OH_DIR   = ROOT_OUT / "preprocessed_features_OH_U25"
FP_DIR   = ROOT_OUT / "preprocessed_features_FP_U25"
TODAY    = dt.date.today().strftime("%Y%m%d")


# ---------- Utilities ----------
def newest_csv(dirpath: Path, pattern: str = "*_cluster_k_summary_*.csv") -> Path | None:
    if not dirpath.exists():
        return None
    files = list(dirpath.glob(pattern))
    if not files:
        return None
    files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return files[0]


def read_ksum(path: Path, dataset_tag: str) -> pd.DataFrame:
    """Expect at least columns: mode, index, k"""
    df = pd.read_csv(path)
    missing = [c for c in ("mode", "index", "k") if c not in df.columns]
    if missing:
        raise ValueError(f"Unexpected columns in {path}. Missing: {missing}")
    df = df.copy()
    df["dataset"] = dataset_tag
    return df


def ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)


def order_categories(df: pd.DataFrame) -> pd.DataFrame:
    mode_order  = ["top3", "cumeig"]
    index_order = ["silhouette", "dunn", "gap", "ch", "db", "ptbiserial"]
    dataset_order = ["OH", "FP"]

    df = df.copy()
    df["mode"]    = pd.Categorical(df["mode"], categories=mode_order, ordered=True)
    df["index"]   = pd.Categorical(df["index"], categories=index_order, ordered=True)
    df["dataset"] = pd.Categorical(df["dataset"], categories=dataset_order, ordered=True)
    return df


# ---------- Figures ----------
def save_heatmap(ksum_all: pd.DataFrame, out_pdf: Path) -> None:
    """
    Heatmap: y=index, x=mode, fill=k, facet by dataset (two panels side-by-side).
    """
    ensure_dir(out_pdf.parent)

    datasets = [d for d in ["OH", "FP"] if d in ksum_all["dataset"].unique().tolist()]
    n_panels = max(1, len(datasets))

    fig, axes = plt.subplots(1, n_panels, figsize=(9, 4.8), constrained_layout=True)
    if n_panels == 1:
        axes = [axes]

    idx_levels = ["silhouette", "dunn", "gap", "ch", "db", "ptbiserial"]
    mode_levels = ["top3", "cumeig"]

    for ax, dset in zip(axes, datasets):
        sub = ksum_all[(ksum_all["dataset"] == dset) & (~ksum_all["k"].isna())]
        mat = pd.pivot_table(sub, index="index", columns="mode", values="k", aggfunc="first")
        mat = mat.reindex(index=idx_levels, columns=mode_levels)

        im = ax.imshow(mat.values.astype(float), aspect="auto")
        # annotate
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                val = mat.iat[i, j]
                text = "" if pd.isna(val) else str(int(val))
                ax.text(j, i, text, ha="center", va="center")

        ax.set_xticks(range(len(mode_levels)))
        ax.set_xticklabels(mode_levels)
        ax.set_yticks(range(len(idx_levels)))
        ax.set_yticklabels(idx_levels)
        ax.set_title(f"Chosen k — {dset}")
        ax.set_xlabel("Dimension mode")
        ax.set_ylabel("Index")

    cbar = fig.colorbar(im, ax=axes, fraction=0.046, pad=0.04)
    cbar.set_label("k")
    fig.suptitle("Chosen number of clusters (k) by index and dimension mode", fontsize=12, fontweight="bold")

    fig.savefig(out_pdf, format="pdf")
    plt.close(fig)


def save_bar(ksum_all: pd.DataFrame, out_pdf: Path) -> None:
    """
    Bar chart: x=index, y=k, facet grid dataset (rows) × mode (cols).
    """
    ensure_dir(out_pdf.parent)

    idx_levels  = ["silhouette", "dunn", "gap", "ch", "db", "ptbiserial"]
    mode_levels = ["top3", "cumeig"]
    dsets       = [d for d in ["OH", "FP"] if d in ksum_all["dataset"].unique().tolist()]

    n_rows = max(1, len(dsets))
    n_cols = len(mode_levels)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 6), sharey=True, constrained_layout=True)
    if n_rows == 1:
        axes = np.array([axes])
    if n_cols == 1:
        axes = axes.reshape(n_rows, 1)

    for r, dset in enumerate(dsets):
        for c, mode in enumerate(mode_levels):
            ax = axes[r, c]
            sub = ksum_all[(ksum_all["dataset"] == dset) & (ksum_all["mode"] == mode) & (~ksum_all["k"].isna())]
            sub = sub.set_index("index").reindex(idx_levels).reset_index()

            ax.bar(sub["index"], sub["k"])
            ax.set_title(f"{dset} — {mode}")
            ax.set_xlabel("Index")
            if c == 0:
                ax.set_ylabel("k")
            ax.set_xticklabels(sub["index"], rotation=25, ha="right")

    fig.suptitle("Chosen number of clusters (k) by index", fontsize=12, fontweight="bold")
    fig.savefig(out_pdf, format="pdf")
    plt.close(fig)


# ---------- Main ----------
def main():
    # find latest k-summary csvs
    oh_csv = newest_csv(OH_DIR)
    fp_csv = newest_csv(FP_DIR)

    if oh_csv is None or fp_csv is None:
        sys.exit(
            "k-summary CSV not found.\n"
            f"  looked in: {OH_DIR}\n"
            f"         and: {FP_DIR}\n"
            "Please run STEP2 to generate per-dataset summaries first."
        )

    print(f"[Found] OH k-summary: {oh_csv}")
    print(f"[Found] FP k-summary: {fp_csv}")

    oh_df = read_ksum(oh_csv, "OH")
    fp_df = read_ksum(fp_csv, "FP")
    ksum_all = pd.concat([oh_df, fp_df], ignore_index=True)

    # order categories and sanitize
    ksum_all = order_categories(ksum_all)

    # outputs
    ensure_dir(ROOT_OUT)
    out_csv_long = ROOT_OUT / f"U25_k_summary_all_long_{TODAY}.csv"
    out_csv_wide = ROOT_OUT / f"U25_k_summary_all_wide_{TODAY}.csv"
    out_pdf_heat = ROOT_OUT / f"Fig_U25_k_heatmap_{TODAY}.pdf"
    out_pdf_bar  = ROOT_OUT / f"Fig_U25_k_bar_{TODAY}.pdf"

    # write long csv
    ksum_all.to_csv(out_csv_long, index=False)
    print(f"[Saved] {out_csv_long}")

    # wide table
    wide = (
        ksum_all
        .loc[:, ["dataset", "mode", "index", "k"]]
        .pivot_table(index="index", columns=["dataset", "mode"], values="k", aggfunc="first")
        .reindex(index=["silhouette", "dunn", "gap", "ch", "db", "ptbiserial"])
        .sort_index()
    )
    wide.columns = [f"{c[0]}_{c[1]}" for c in wide.columns.to_flat_index()]
    wide.to_csv(out_csv_wide)
    print(f"[Saved] {out_csv_wide}")

    # console preview
    print("\n--- Console preview: U25_k_summary_all_wide ---")
    print(wide.to_string())

    # figures
    save_heatmap(ksum_all, out_pdf_heat)
    print(f"[Saved] {out_pdf_heat}")

    save_bar(ksum_all, out_pdf_bar)
    print(f"[Saved] {out_pdf_bar}")

    print("\n✅ Done. Consolidated CSVs and PDFs are under:")
    print(ROOT_OUT)


if __name__ == "__main__":
    main()


[Found] OH k-summary: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/preprocessed_features_OH_U25/preprocessed_features_OH_U25_cluster_k_summary_20251026.csv
[Found] FP k-summary: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/preprocessed_features_FP_U25/preprocessed_features_FP_U25_cluster_k_summary_20251026.csv
[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/U25_k_summary_all_long_20251026.csv


/tmp/ipykernel_430862/4078340635.py:225: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .pivot_table(index="index", columns=["dataset", "mode"], values="k", aggfunc="first")
/tmp/ipykernel_430862/4078340635.py:122: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  mat = pd.pivot_table(sub, index="index", columns="mode", values="k", aggfunc="first")
/tmp/ipykernel_430862/4078340635.py:122: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  mat = pd.pivot_table(sub, index="index", columns="mode", values="k", aggfun

[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/U25_k_summary_all_wide_20251026.csv

--- Console preview: U25_k_summary_all_wide ---
            OH_top3  OH_cumeig  FP_top3  FP_cumeig
index                                             
ch               25          2       25         25
db                3          3        3          3
dunn              2         14        5          2
gap               2          2        2          2
ptbiserial        5         25        6          6
silhouette        2          2       23         25
[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/Fig_U25_k_heatmap_20251026.pdf


/tmp/ipykernel_430862/4078340635.py:179: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(sub["index"], rotation=25, ha="right")
/tmp/ipykernel_430862/4078340635.py:179: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(sub["index"], rotation=25, ha="right")
/tmp/ipykernel_430862/4078340635.py:179: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(sub["index"], rotation=25, ha="right")
/tmp/ipykernel_430862/4078340635.py:179: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(sub["index"], rotation=25, ha="right")


[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/Fig_U25_k_bar_20251026.pdf

✅ Done. Consolidated CSVs and PDFs are under:
/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25


In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
paper_STEP2_U25_k_summary_build.py
----------------------------------
Consolidate and publish paper-ready outputs (CSV + PDF) for the fixed "≤25 clusters" selection (STEP2_U25),
AND (optionally) plot index profiles: x = number of clusters (k), y = index score, with chosen k highlighted.

[Inputs] (auto-detected "latest" files):
  ROOT = /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25
  ├─ preprocessed_features_OH_U25/
  │    ├─ *_cluster_k_summary_*.csv      ← required (latest used)
  │    └─ (optional) index profiles like:
  │         index_profile_top3_silhouette.csv     # columns: k, score
  │         index_profile_top3_dunn.csv
  │         index_profile_cumeig_gap.csv
  │         ... (index_profile_{mode}_{index}.csv)
  └─ preprocessed_features_FP_U25/
       ├─ *_cluster_k_summary_*.csv      ← required (latest used)
       └─ (optional) same pattern as above

  Expected columns in each k-summary CSV:  mode, index, k
  Expected columns in each index_profile CSV:  k, score

[Outputs] (written directly under ROOT):
  ├─ U25_k_summary_all_long_YYYYMMDD.csv
  ├─ U25_k_summary_all_wide_YYYYMMDD.csv
  ├─ Fig_U25_k_heatmap_YYYYMMDD.pdf
  ├─ Fig_U25_k_bar_YYYYMMDD.pdf
  └─ profiles/
       ├─ Fig_U25_index_profiles_OH_top3_YYYYMMDD.pdf        # x=k, y=index score; one panel per index
       ├─ Fig_U25_index_profiles_OH_cumeig_YYYYMMDD.pdf
       ├─ Fig_U25_index_profiles_FP_top3_YYYYMMDD.pdf
       └─ Fig_U25_index_profiles_FP_cumeig_YYYYMMDD.pdf

[Console]
  - Prints the wide table preview.
  - For each dataset×mode×index: prints "chosen k" and, if profiles exist, the best k and score.

[Figure/Text policy]
  - All labels/titles in figures are **English only**.
"""

from __future__ import annotations
import sys
from pathlib import Path
import datetime as dt
from typing import Optional, Dict, Tuple

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# ---------- Paths ----------
BASE_DIR = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No")
ROOT_OUT = BASE_DIR / "20251026_under_25clusters" / "STEP2_U25"
OH_DIR   = ROOT_OUT / "preprocessed_features_OH_U25"
FP_DIR   = ROOT_OUT / "preprocessed_features_FP_U25"
PROF_DIR = ROOT_OUT / "profiles"
TODAY    = dt.date.today().strftime("%Y%m%d")


# ---------- Utilities ----------
def newest_csv(dirpath: Path, pattern: str = "*_cluster_k_summary_*.csv") -> Optional[Path]:
    if not dirpath.exists():
        return None
    files = list(dirpath.glob(pattern))
    if not files:
        return None
    files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return files[0]


def read_ksum(path: Path, dataset_tag: str) -> pd.DataFrame:
    """Expect at least columns: mode, index, k"""
    df = pd.read_csv(path)
    missing = [c for c in ("mode", "index", "k") if c not in df.columns]
    if missing:
        raise ValueError(f"Unexpected columns in {path}. Missing: {missing}")
    df = df.copy()
    df["dataset"] = dataset_tag
    return df


def ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)


def order_categories(df: pd.DataFrame) -> pd.DataFrame:
    mode_order  = ["top3", "cumeig"]
    index_order = ["silhouette", "dunn", "gap", "ch", "db", "ptbiserial"]
    dataset_order = ["OH", "FP"]

    df = df.copy()
    df["mode"]    = pd.Categorical(df["mode"], categories=mode_order, ordered=True)
    df["index"]   = pd.Categorical(df["index"], categories=index_order, ordered=True)
    df["dataset"] = pd.Categorical(df["dataset"], categories=dataset_order, ordered=True)
    return df


# ---------- Figures (k summary) ----------
def save_heatmap(ksum_all: pd.DataFrame, out_pdf: Path) -> None:
    """Heatmap: y=index, x=mode, fill=k, facet by dataset (two panels side-by-side)."""
    ensure_dir(out_pdf.parent)

    datasets = [d for d in ["OH", "FP"] if d in ksum_all["dataset"].unique().tolist()]
    n_panels = max(1, len(datasets))

    fig, axes = plt.subplots(1, n_panels, figsize=(9, 4.8), constrained_layout=True)
    if n_panels == 1:
        axes = [axes]

    idx_levels = ["silhouette", "dunn", "gap", "ch", "db", "ptbiserial"]
    mode_levels = ["top3", "cumeig"]

    for ax, dset in zip(axes, datasets):
        sub = ksum_all[(ksum_all["dataset"] == dset) & (~ksum_all["k"].isna())]
        mat = pd.pivot_table(sub, index="index", columns="mode", values="k", aggfunc="first")
        mat = mat.reindex(index=idx_levels, columns=mode_levels)

        im = ax.imshow(mat.values.astype(float), aspect="auto")
        # annotate
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                val = mat.iat[i, j]
                text = "" if pd.isna(val) else str(int(val))
                ax.text(j, i, text, ha="center", va="center")

        ax.set_xticks(range(len(mode_levels)))
        ax.set_xticklabels(mode_levels)
        ax.set_yticks(range(len(idx_levels)))
        ax.set_yticklabels(idx_levels)
        ax.set_title(f"Chosen k — {dset}")
        ax.set_xlabel("Dimension mode")
        ax.set_ylabel("Index")

    cbar = fig.colorbar(im, ax=axes, fraction=0.046, pad=0.04)
    cbar.set_label("k")
    fig.suptitle("Chosen number of clusters (k) by index and dimension mode", fontsize=12, fontweight="bold")

    fig.savefig(out_pdf, format="pdf")
    plt.close(fig)


def save_bar(ksum_all: pd.DataFrame, out_pdf: Path) -> None:
    """Bar chart: x=index, y=k, facet grid dataset (rows) × mode (cols)."""
    ensure_dir(out_pdf.parent)

    idx_levels  = ["silhouette", "dunn", "gap", "ch", "db", "ptbiserial"]
    mode_levels = ["top3", "cumeig"]
    dsets       = [d for d in ["OH", "FP"] if d in ksum_all["dataset"].unique().tolist()]

    n_rows = max(1, len(dsets))
    n_cols = len(mode_levels)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 6), sharey=True, constrained_layout=True)
    if n_rows == 1:
        axes = np.array([axes])
    if n_cols == 1:
        axes = axes.reshape(n_rows, 1)

    for r, dset in enumerate(dsets):
        for c, mode in enumerate(mode_levels):
            ax = axes[r, c]
            sub = ksum_all[(ksum_all["dataset"] == dset) & (ksum_all["mode"] == mode) & (~ksum_all["k"].isna())]
            sub = sub.set_index("index").reindex(idx_levels).reset_index()

            ax.bar(sub["index"], sub["k"])
            ax.set_title(f"{dset} — {mode}")
            ax.set_xlabel("Index")
            if c == 0:
                ax.set_ylabel("k")
            ax.set_xticklabels(sub["index"], rotation=25, ha="right")

    fig.suptitle("Chosen number of clusters (k) by index", fontsize=12, fontweight="bold")
    fig.savefig(out_pdf, format="pdf")
    plt.close(fig)


# ---------- Optional Index Profile Loader ----------
def load_index_profile(dir_path: Path, mode: str, index_name: str) -> Optional[pd.DataFrame]:
    """
    Try to load optional profile CSV: index_profile_{mode}_{index}.csv
    - Expected columns: k, score
    - Returns DataFrame sorted by k, or None if not found.
    """
    cand = dir_path / f"index_profile_{mode}_{index_name}.csv"
    if not cand.exists():
        return None
    df = pd.read_csv(cand)
    if not {"k", "score"}.issubset(df.columns):
        return None
    df = df.sort_values("k").reset_index(drop=True)
    return df


def best_k_from_profile(df: pd.DataFrame, index_name: str) -> Tuple[int, float]:
    """
    Return (k*, score*) for the profile, choosing max(score) for
    indices where higher is better (silhouette, ch, gap, ptbiserial)
    and min(score) for DB / Dunn?  NOTE: Dunn: higher is better; DB: lower is better.
    """
    if index_name in {"db"}:
        # Davies-Bouldin: lower is better
        idx = df["score"].idxmin()
    else:
        # silhouette, dunn, gap, ch, ptbiserial: higher is better
        idx = df["score"].idxmax()
    row = df.loc[idx]
    return int(row["k"]), float(row["score"])


# ---------- Figures (Index Profiles) ----------
def save_index_profiles(
    ksum_all: pd.DataFrame,
    oh_dir: Path,
    fp_dir: Path,
    out_dir: Path
) -> None:
    """
    Create per-dataset, per-mode multi-panel PDFs:
      - x = k (2..25), y = index score (if profile CSV exists)
      - annotate vertical line at chosen k from ksum_all
      - if profile is missing, draw only the chosen k as a single marker
    """
    ensure_dir(out_dir)

    idx_levels  = ["silhouette", "dunn", "gap", "ch", "db", "ptbiserial"]
    mode_levels = ["top3", "cumeig"]
    dset_dirs   = {"OH": oh_dir, "FP": fp_dir}

    for dset, ddir in dset_dirs.items():
        for mode in mode_levels:
            sub = (
                ksum_all[(ksum_all["dataset"] == dset) & (ksum_all["mode"] == mode)]
                .set_index("index")
                .reindex(idx_levels)
            )
            # Prepare figure with 6 panels (one per index)
            fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
            axes = axes.ravel()

            for ax, index_name in zip(axes, idx_levels):
                chosen_k = sub.loc[index_name, "k"] if index_name in sub.index else np.nan
                # Try load profile (k, score)
                prof = load_index_profile(ddir, mode, index_name)

                if prof is not None and len(prof) > 0:
                    ax.plot(prof["k"], prof["score"])
                    # Mark the best according to index rule
                    k_star, score_star = best_k_from_profile(prof, index_name)
                    ax.scatter([k_star], [score_star], marker="o")
                    # Also mark chosen_k from NbClust summary, if available
                    if pd.notna(chosen_k):
                        # find y for chosen_k to place a marker; interpolate if missing
                        if (prof["k"] == chosen_k).any():
                            y_ch = float(prof.loc[prof["k"] == chosen_k, "score"].iloc[0])
                        else:
                            # simple interpolation if possible
                            y_ch = np.interp(chosen_k, prof["k"], prof["score"])
                        ax.axvline(chosen_k, linestyle="--")
                        ax.scatter([chosen_k], [y_ch], marker="x")
                        ax.legend(["profile", "best(k*)", "chosen k"], loc="best")
                    ax.set_title(f"{index_name}")
                    ax.set_xlabel("Clusters (k)")
                    ax.set_ylabel("Index score")
                else:
                    # Fallback: no profile available, just show chosen k if present
                    ax.set_title(f"{index_name} (no profile)")
                    ax.set_xlabel("Clusters (k)")
                    ax.set_ylabel("Index score")
                    if pd.notna(chosen_k):
                        ax.axvline(chosen_k, linestyle="--")
                        ax.text(0.5, 0.5, f"chosen k = {int(chosen_k)}",
                                ha="center", va="center", transform=ax.transAxes)

            fig.suptitle(f"Index profiles — {dset} — {mode}", fontsize=12, fontweight="bold")
            out_pdf = out_dir / f"Fig_U25_index_profiles_{dset}_{mode}_{TODAY}.pdf"
            fig.savefig(out_pdf, format="pdf")
            plt.close(fig)
            print(f"[Saved] {out_pdf}")


# ---------- Main ----------
def main():
    # find latest k-summary csvs
    oh_csv = newest_csv(OH_DIR)
    fp_csv = newest_csv(FP_DIR)

    if oh_csv is None or fp_csv is None:
        sys.exit(
            "k-summary CSV not found.\n"
            f"  looked in: {OH_DIR}\n"
            f"         and: {FP_DIR}\n"
            "Please run STEP2 to generate per-dataset summaries first."
        )

    print(f"[Found] OH k-summary: {oh_csv}")
    print(f"[Found] FP k-summary: {fp_csv}")

    oh_df = read_ksum(oh_csv, "OH")
    fp_df = read_ksum(fp_csv, "FP")
    ksum_all = pd.concat([oh_df, fp_df], ignore_index=True)

    # order categories and sanitize
    ksum_all = order_categories(ksum_all)

    # outputs (CSV + base figs)
    ensure_dir(ROOT_OUT)
    ensure_dir(PROF_DIR)

    out_csv_long = ROOT_OUT / f"U25_k_summary_all_long_{TODAY}.csv"
    out_csv_wide = ROOT_OUT / f"U25_k_summary_all_wide_{TODAY}.csv"
    out_pdf_heat = ROOT_OUT / f"Fig_U25_k_heatmap_{TODAY}.pdf"
    out_pdf_bar  = ROOT_OUT / f"Fig_U25_k_bar_{TODAY}.pdf"

    # write long csv
    ksum_all.to_csv(out_csv_long, index=False)
    print(f"[Saved] {out_csv_long}")

    # wide table
    wide = (
        ksum_all
        .loc[:, ["dataset", "mode", "index", "k"]]
        .pivot_table(index="index", columns=["dataset", "mode"], values="k", aggfunc="first")
        .reindex(index=["silhouette", "dunn", "gap", "ch", "db", "ptbiserial"])
        .sort_index()
    )
    # flatten multiindex columns to dataset_mode
    if isinstance(wide.columns, pd.MultiIndex):
        wide.columns = [f"{c[0]}_{c[1]}" for c in wide.columns.to_flat_index()]
    wide.to_csv(out_csv_wide)
    print(f"[Saved] {out_csv_wide}")

    # console preview (wide)
    print("\n--- Console preview: U25_k_summary_all_wide ---")
    print(wide.to_string())

    # console details per dataset×mode×index (chosen k, best from profile if available)
    idx_levels  = ["silhouette", "dunn", "gap", "ch", "db", "ptbiserial"]
    mode_levels = ["top3", "cumeig"]
    print("\n--- Console: chosen k and (if available) best k from profile ---")
    for dset, ddir in [("OH", OH_DIR), ("FP", FP_DIR)]:
        for mode in mode_levels:
            for index_name in idx_levels:
                chosen_k = ksum_all.query("dataset == @dset and mode == @mode and index == @index_name")["k"]
                chosen_k_val = int(chosen_k.iloc[0]) if len(chosen_k) and not pd.isna(chosen_k.iloc[0]) else None
                prof = load_index_profile(ddir, mode, index_name)
                if prof is not None and len(prof) > 0:
                    k_star, score_star = best_k_from_profile(prof, index_name)
                    if chosen_k_val is None:
                        print(f"{dset}/{mode}/{index_name}: best k*={k_star}, score*={score_star:.4g} | chosen k=NA")
                    else:
                        print(f"{dset}/{mode}/{index_name}: best k*={k_star}, score*={score_star:.4g} | chosen k={chosen_k_val}")
                else:
                    if chosen_k_val is None:
                        print(f"{dset}/{mode}/{index_name}: (no profile) | chosen k=NA")
                    else:
                        print(f"{dset}/{mode}/{index_name}: (no profile) | chosen k={chosen_k_val}")

    # figures
    save_heatmap(ksum_all, out_pdf_heat)
    print(f"[Saved] {out_pdf_heat}")

    save_bar(ksum_all, out_pdf_bar)
    print(f"[Saved] {out_pdf_bar}")

    # index profile figures (optional, uses profiles if present; otherwise marks chosen k)
    save_index_profiles(ksum_all, OH_DIR, FP_DIR, PROF_DIR)

    print("\n✅ Done. Consolidated CSVs and PDFs are under:")
    print(" -", ROOT_OUT)
    print(" -", PROF_DIR)


if __name__ == "__main__":
    main()


[Found] OH k-summary: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/preprocessed_features_OH_U25/preprocessed_features_OH_U25_cluster_k_summary_20251026.csv
[Found] FP k-summary: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/preprocessed_features_FP_U25/preprocessed_features_FP_U25_cluster_k_summary_20251026.csv
[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/U25_k_summary_all_long_20251026.csv
[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/U25_k_summary_all_wide_20251026.csv

--- Console preview: U25_k_summary_all_wide ---
            OH_top3  OH_cumeig  FP_top3  FP_cumeig
index                                             
ch               25          2       25         25
db                3          3        3          3
dunn              2         14        5          2
ga

/tmp/ipykernel_430862/3540007257.py:326: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .pivot_table(index="index", columns=["dataset", "mode"], values="k", aggfunc="first")


FP/cumeig/dunn: (no profile) | chosen k=2
FP/cumeig/gap: (no profile) | chosen k=2
FP/cumeig/ch: (no profile) | chosen k=25
FP/cumeig/db: (no profile) | chosen k=3
FP/cumeig/ptbiserial: (no profile) | chosen k=6


/tmp/ipykernel_430862/3540007257.py:119: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  mat = pd.pivot_table(sub, index="index", columns="mode", values="k", aggfunc="first")
/tmp/ipykernel_430862/3540007257.py:119: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  mat = pd.pivot_table(sub, index="index", columns="mode", values="k", aggfunc="first")


[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/Fig_U25_k_heatmap_20251026.pdf


/tmp/ipykernel_430862/3540007257.py:174: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(sub["index"], rotation=25, ha="right")
/tmp/ipykernel_430862/3540007257.py:174: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(sub["index"], rotation=25, ha="right")
/tmp/ipykernel_430862/3540007257.py:174: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(sub["index"], rotation=25, ha="right")
/tmp/ipykernel_430862/3540007257.py:174: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(sub["index"], rotation=25, ha="right")


[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/Fig_U25_k_bar_20251026.pdf
[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/profiles/Fig_U25_index_profiles_OH_top3_20251026.pdf
[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/profiles/Fig_U25_index_profiles_OH_cumeig_20251026.pdf
[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/profiles/Fig_U25_index_profiles_FP_top3_20251026.pdf
[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/profiles/Fig_U25_index_profiles_FP_cumeig_20251026.pdf

✅ Done. Consolidated CSVs and PDFs are under:
 - /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25
 - /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/ST